In [48]:
# Importations
import os
import psycopg
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import round,col, dayofmonth,month,year,quarter,when 
from pyspark.sql import functions as F

In [49]:
# Chargement des variables d'environnement (.env)
load_dotenv()


True

In [50]:
# Paramètres de connexion et chemins
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

JDBC_URL = f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME}"
JDBC_PROPS = {
    "user": DB_USER,
    "password": DB_PASSWORD,
    "driver": "org.postgresql.Driver"
}

In [51]:
# Initialisation de la session Spark
spark = SparkSession.builder \
    .appName("WindPowerSilverTransformation") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

In [52]:
# Récupère la table windturbinepower_raw depuis PostgreSQL dans un DataFrame Spark
windturbinepower_raw_df = spark.read.jdbc(
    url=JDBC_URL,
    table='bronze.windturbinepower_raw',
    properties=JDBC_PROPS
)

In [53]:
# Nettoyage et enrichissement des données
# Transformation: arrondir les colonnes "energy_produced" et "wind_speed"
# Transformation: créer la colonne "day" à partir de "date" avec la méthode "dayofmonth"
# Transformation: créer la colonne "month" à partir de "date" avec la méthode "month"
# Transformation: créer la colonne "quarter" à partir de "date" avec la méthode "quarter"
# Transformation: créer la colonne "year" à partir de "date" avec la méthode "year"
# Transformation: formater la colonne "time" au format "HH:mm:ss"
# Transformation: créer les colonnes "hour_of_day", "minute_of_hour", "second_of_minute" à partir de "time"

df_transformed = (
    windturbinepower_raw_df
    .withColumn("wind_speed", round(col("wind_speed"), 2))
    .withColumn("energy_produced", round(col("energy_produced"), 2))
    .withColumn("day", dayofmonth(col("date")))
    .withColumn("month", month(col("date")))
    .withColumn("quarter", quarter(col("date")))
    .withColumn("year", year(col("date")))
    .withColumn("time", F.date_format(col("time"), "HH:mm:ss"))
    .withColumn("hour_of_day", F.hour(col("time")))
    .withColumn("minute_of_hour", F.minute(col("time")))
    .withColumn("second_of_minute", F.second(col("time")))
    .withColumn(
        "time_period",
        when((col("hour_of_day") >= 5) & (col("hour_of_day") < 12), "Morning")
        .when((col("hour_of_day") >= 12) & (col("hour_of_day") < 17), "Afternoon")
        .when((col("hour_of_day") >= 17) & (col("hour_of_day") < 21), "Evening")
        .otherwise("Night")
    )
)

# Suppression des colonnes 
df_transformed = df_transformed.drop("ingested_at", "source_file", "batch_id","measured_at")

df_transformed.show(5)

+-------------+----------+--------+------------+--------+-------------+--------+---------+--------+------+----------------------+----------+--------------+---------------+---+-----+-------+----+-----------+--------------+----------------+-----------+
|production_id|      date|    time|turbine_name|capacity|location_name|latitude|longitude|  region|status|responsible_department|wind_speed|wind_direction|energy_produced|day|month|quarter|year|hour_of_day|minute_of_hour|second_of_minute|time_period|
+-------------+----------+--------+------------+--------+-------------+--------+---------+--------+------+----------------------+----------+--------------+---------------+---+-----+-------+----+-----------+--------------+----------------+-----------+
|         6481|2024-06-16|00:00:00|   Turbine A|    2200|   Location 1| 34.0522|-118.2437|Region A|Online|            Operations|     17.49|             W|        1878.96| 16|    6|      2|2024|          0|             0|               0|      Nig

In [54]:
# Définition de la table silver.windpowerturbine_cleaned avec les types de données appropriés
create_table_sql = """
CREATE TABLE IF NOT EXISTS silver.windpowerturbine_cleaned (
    production_id INTEGER,
    date DATE,
    time TIME,
    turbine_name VARCHAR(100),
    capacity DOUBLE PRECISION,
    location_name VARCHAR(100),
    latitude DOUBLE PRECISION,
    longitude DOUBLE PRECISION,
    region VARCHAR(100),
    status VARCHAR(50),
    responsible_department VARCHAR(100),
    wind_speed DOUBLE PRECISION,
    wind_direction VARCHAR(50),
    energy_produced DOUBLE PRECISION,
    day INTEGER,
    month INTEGER,
    quarter INTEGER,
    year INTEGER,
    hour_of_day INTEGER,
    minute_of_hour INTEGER,
    second_of_minute INTEGER,
    time_period VARCHAR(20)
);
"""

# Connexion à PostgreSQL et exécution du script de création de table
with psycopg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    autocommit=True
) as conn:
    with conn.cursor() as cur:
        # Crée le schéma s'il n'existe pas
        cur.execute("CREATE SCHEMA IF NOT EXISTS silver;")
        
        # Crée la table avec script SQL
        cur.execute(create_table_sql)
        print("Schéma silver et table windpowerturbine_cleaned créés avec les types corrects")
        
        # Ajoute la clé primaire après création de la table
        try:
            cur.execute("""
                ALTER TABLE silver.windpowerturbine_cleaned
                ADD CONSTRAINT windpowerturbine_cleaned_pkey PRIMARY KEY (production_id);
            """)
            print("Clé primaire ajoutée sur (production_id)")
        except (psycopg.errors.DuplicateTable, psycopg.errors.DuplicateObject, psycopg.errors.InvalidTableDefinition):
            print("Clé primaire existe déjà")
        


Schéma silver et table windpowerturbine_cleaned créés avec les types corrects
Clé primaire existe déjà


In [55]:
# Écriture dans silver.windpowerturbine_cleaned avec protection anti-doublons

total_count = df_transformed.count()
print(f"Lignes à traiter: {total_count}")

# Filtrage des doublons (basé sur la clé primaire: production_id)
try:
    existing_df = spark.read.jdbc(url=JDBC_URL, table='silver.windpowerturbine_cleaned', properties=JDBC_PROPS)
    
    existing_keys = existing_df.select("production_id")
    df_to_insert = df_transformed.join(existing_keys, on=["production_id"], how="left_anti")
    
    # Sécurité supplémentaire: le dataframe résultat récupéré peut contenir des doublons internes, on les élimine avant insertion
    df_to_insert = df_to_insert.dropDuplicates(["production_id"])
    new_count = df_to_insert.count()
    print(f"Lignes existantes: {existing_df.count()} | Nouvelles: {new_count} | Doublons ignorés: {total_count - new_count}")
    
except Exception:
    df_to_insert = df_transformed.dropDuplicates(["production_id"])
    new_count = df_to_insert.count()
    print(f"Table vide -> Insertion de {new_count} lignes après déduplication")

# Insertion triée par date, time, turbine_name
if new_count > 0:
    df_final = df_to_insert \
        .withColumn("time", F.to_timestamp(col("time"), "HH:mm:ss")) \
        .orderBy("date", "time", "turbine_name")
    
    df_final.write.jdbc(url=JDBC_URL, table="silver.windpowerturbine_cleaned", mode="append", properties=JDBC_PROPS)
    print(f"{new_count} lignes insérées avec succès")
else:
    print("Aucune donnée à insérer")

# Vérification finale
final_count = spark.read.jdbc(url=JDBC_URL, table='silver.windpowerturbine_cleaned', properties=JDBC_PROPS).count()
print(f"\nTotal en base: {final_count} lignes")

Lignes à traiter: 432
Lignes existantes: 432 | Nouvelles: 432 | Doublons ignorés: 0
432 lignes insérées avec succès

Total en base: 864 lignes
